# **Gold (Sales)**

In [0]:
USE CATALOG smart_odoo;
drop table gold.fact_sales;
CREATE TABLE IF NOT EXISTS gold.fact_sales
(
    sale_order_line_id  INT,
    order_id            INT,
    product_id          int,
    customer_id         int,
    order_state         STRING,
    invoice_status      STRING,
    order_date          TIMESTAMP,
    product_qty         DOUBLE,
    price_unit          DOUBLE,
    discount            DOUBLE,
    price_subtotal      DOUBLE,
    price_total         DOUBLE,
    price_tax           DOUBLE,
    net_revenue         DOUBLE,
    gross_revenue       DOUBLE,
    created_at          TIMESTAMP,
    updated_at          TIMESTAMP,
    dwh_loaded_at       TIMESTAMP
);

MERGE INTO gold.fact_sales AS target
USING (
    SELECT
        cast(sol.sale_order_line_id AS INT),
        cast(sol.sale_order_id AS INT)                                       AS order_id,
        cast(sol.product_id AS INT),
        cast(sol.partner_id  AS INT)                                       AS customer_id,
        so.order_state,
        sol.invoice_status,
        CAST(so.date_order AS DATE)                             AS order_date,
        sol.product_uom_qty                                     AS product_qty,
        sol.price_unit,
        sol.discount,
        sol.price_subtotal,
        sol.price_total,
        sol.price_tax,
        CASE
            WHEN so.order_state IN ('sale', 'done')
            THEN sol.price_subtotal
            ELSE 0
        END                                                     AS net_revenue,
        CASE
            WHEN so.order_state IN ('sale', 'done')
            THEN sol.price_total
            ELSE 0
        END                                                     AS gross_revenue,
        sol.created_at,
        sol.updated_at,
        current_timestamp()                                     AS dwh_loaded_at

    FROM silver.sale_order_line sol
    JOIN silver.sale_order so
        ON sol.sale_order_id = so.sale_order_id

    WHERE sol.updated_at >= COALESCE(
        (SELECT MAX(updated_at) FROM gold.fact_sales),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.sale_order_line_id = source.sale_order_line_id

WHEN MATCHED AND target.updated_at < source.updated_at
    THEN UPDATE SET
        target.order_id         = source.order_id,
        target.product_id       = source.product_id,
        target.customer_id      = source.customer_id,
        target.order_state      = source.order_state,
        target.invoice_status   = source.invoice_status,
        target.order_date       = source.order_date,
        target.product_qty      = source.product_qty,
        target.price_unit       = source.price_unit,
        target.discount         = source.discount,
        target.price_subtotal   = source.price_subtotal,
        target.price_total      = source.price_total,
        target.price_tax        = source.price_tax,
        target.net_revenue      = source.net_revenue,
        target.gross_revenue    = source.gross_revenue,
        target.created_at       = source.created_at,
        target.updated_at       = source.updated_at,
        target.dwh_loaded_at    = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    sale_order_line_id,
    order_id,
    product_id,
    customer_id,
    order_state,
    invoice_status,
    order_date,
    product_qty,
    price_unit,
    discount,
    price_subtotal,
    price_total,
    price_tax,
    net_revenue,
    gross_revenue,
    created_at,
    updated_at,
    dwh_loaded_at
)
VALUES (
    source.sale_order_line_id,
    source.order_id,
    source.product_id,
    source.customer_id,
    source.order_state,
    source.invoice_status,
    source.order_date,
    source.product_qty,
    source.price_unit,
    source.discount,
    source.price_subtotal,
    source.price_total,
    source.price_tax,
    source.net_revenue,
    source.gross_revenue,
    source.created_at,
    source.updated_at,
    current_timestamp()
);